# 03 — Agent IA 1 V4 : comparaison multi-échelle autonome

Le LLM **propose librement** une fenêtre et un pas. Python génère des perturbations **relatives** de cette proposition, exécute les détecteurs existants et mesure couverture, fragmentation, confiance, coût et recouvrement spatial. Le LLM interprète ces observations et décide : **accept**, **select** ou **revise**.

**Important :** aucun score ne constitue une mesure de précision ou de rappel sans annotations de référence. Une échelle stable n'est pas nécessairement correcte. Les seuils des détecteurs de la V3 sont conservés. Trois cycles de comparaison au maximum ; les décisions et tous les essais sont tracés.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import requests

## 1. Charger la route normalisée du notebook 02 V2

In [ ]:
ROUTE_PATH = Path("data/output/route_resampled.csv")

if not ROUTE_PATH.exists():
    raise FileNotFoundError(
        "Exécute d'abord le notebook 02 V2."
    )

route = pd.read_csv(ROUTE_PATH)

required = {"distance_m", "lat", "lon", "elev_m"}
missing = required - set(route.columns)

if missing:
    raise ValueError(
        f"Colonnes manquantes : {sorted(missing)}"
    )

BASE_SPACING_M = float(
    np.median(
        np.diff(route["distance_m"].to_numpy(dtype=float))[:-1]
    )
    if len(route) > 2
    else 10.0
)

ROUTE_LENGTH_M = float(route["distance_m"].iloc[-1])

print("Route :", ROUTE_PATH)
print("Longueur :", round(ROUTE_LENGTH_M, 1), "m")
print("Résolution de base :", round(BASE_SPACING_M, 3), "m")

## 2. Caractéristiques et outils disponibles

Les bornes ci-dessous ne sont **pas des choix proposés au LLM**.

Ce sont uniquement des garde-fous pour empêcher des valeurs absurdes comme :

```text
fenêtre = 0.1 m
fenêtre = 20 km sur une trace de 5 km
```

À l'intérieur de ces bornes, l'agent peut choisir librement n'importe quelle valeur.

In [ ]:
FEATURES = {
    "forte_pente": {
        "family": "slope_intensity",
        "detector": "slope",
        "description": (
            "Identifier des portions où la pente est localement forte. "
            "Le phénomène peut être court et doit conserver les variations marquées."
        ),
        "window_bounds_m": (10.0, 250.0),
    },

    "pente_douce": {
        "family": "slope_intensity",
        "detector": "slope",
        "description": (
            "Identifier des portions de pente modérée et relativement progressive, "
            "sans confondre une oscillation locale avec une tendance réelle."
        ),
        "window_bounds_m": (10.0, 400.0),
    },

    "montee": {
        "family": "slope_direction",
        "detector": "slope",
        "description": (
            "Identifier une tendance de montée significative le long de l'itinéraire. "
            "On cherche davantage une tendance spatiale qu'une micro-variation."
        ),
        "window_bounds_m": (20.0, 600.0),
    },

    "descente": {
        "family": "slope_direction",
        "detector": "slope",
        "description": (
            "Identifier une tendance de descente significative le long de l'itinéraire. "
            "On cherche davantage une tendance spatiale qu'une micro-variation."
        ),
        "window_bounds_m": (20.0, 600.0),
    },

    "lacets": {
        "family": "geometry",
        "detector": "switchbacks",
        "description": (
            "Identifier une succession rapprochée de changements de direction alternés "
            "caractéristiques de lacets. La fenêtre doit être assez grande pour observer "
            "plusieurs virages, mais pas au point de mélanger des zones indépendantes."
        ),
        "window_bounds_m": (30.0, 800.0),
    },
}

## 3. Résumé de la trace fourni à l'agent

L'agent ne reçoit pas tous les points bruts.

On lui fournit un résumé compact utile pour son choix d'échelle.

In [ ]:
def build_route_summary(route):
    d = route["distance_m"].to_numpy(dtype=float)
    z = route["elev_m"].to_numpy(dtype=float)

    dz = np.diff(z)
    dd = np.diff(d)

    valid = dd > 0

    local_slopes = np.zeros_like(dz, dtype=float)
    local_slopes[valid] = 100.0 * dz[valid] / dd[valid]

    return {
        "route_length_m": float(d[-1]),
        "base_spacing_m": float(
            np.median(dd[valid])
            if valid.any()
            else 10.0
        ),
        "elevation_min_m": float(np.min(z)),
        "elevation_max_m": float(np.max(z)),
        "elevation_range_m": float(np.max(z) - np.min(z)),
        "median_abs_base_slope_pct": float(
            np.median(np.abs(local_slopes[valid]))
            if valid.any()
            else 0.0
        ),
        "p90_abs_base_slope_pct": float(
            np.percentile(np.abs(local_slopes[valid]), 90)
            if valid.any()
            else 0.0
        ),
        "n_points": int(len(route)),
    }


ROUTE_SUMMARY = build_route_summary(route)

print(
    json.dumps(
        ROUTE_SUMMARY,
        ensure_ascii=False,
        indent=2,
    )
)

## 4. Détecteur déterministe de pente

L'agent choisit l'échelle.

Python calcule la pente.

In [ ]:
def compute_window_slopes(
    route,
    window_m,
    step_m,
):
    distances = route["distance_m"].to_numpy(dtype=float)
    elevations = route["elev_m"].to_numpy(dtype=float)

    rows = []
    start = 0.0
    total = float(distances[-1])

    while start + window_m <= total + 1e-9:
        end = min(start + window_m, total)

        if end - start < window_m - 1e-6:
            break

        z0 = np.interp(
            start,
            distances,
            elevations,
        )

        z1 = np.interp(
            end,
            distances,
            elevations,
        )

        slope_pct = (
            100.0
            * (z1 - z0)
            / (end - start)
        )

        rows.append({
            "start_m": float(start),
            "end_m": float(end),
            "slope_pct": float(slope_pct),
        })

        start += step_m

    return pd.DataFrame(rows)


def detect_slope_feature(
    route,
    feature,
    window_m,
    step_m,
):
    slopes = compute_window_slopes(
        route,
        window_m=window_m,
        step_m=step_m,
    )

    if slopes.empty:
        return slopes

    abs_slope = slopes["slope_pct"].abs()

    if feature == "forte_pente":
        mask = abs_slope >= 12.0

    elif feature == "pente_douce":
        mask = (
            (abs_slope >= 3.0)
            & (abs_slope < 8.0)
        )

    elif feature == "montee":
        mask = slopes["slope_pct"] >= 5.0

    elif feature == "descente":
        mask = slopes["slope_pct"] <= -5.0

    else:
        raise ValueError(feature)

    out = slopes.loc[mask].copy()

    if out.empty:
        return out

    out["feature"] = feature
    out["family"] = FEATURES[feature]["family"]
    out["resolution_m"] = float(window_m)

    out["confidence"] = np.clip(
        abs(out["slope_pct"]) / 20.0,
        0.0,
        1.0,
    )

    return out

## 5. Détecteur déterministe de lacets

In [ ]:
def bearing_deg(
    lat1,
    lon1,
    lat2,
    lon2,
):
    phi1 = math.radians(float(lat1))
    phi2 = math.radians(float(lat2))

    dlambda = math.radians(
        float(lon2) - float(lon1)
    )

    y = (
        math.sin(dlambda)
        * math.cos(phi2)
    )

    x = (
        math.cos(phi1)
        * math.sin(phi2)
        - math.sin(phi1)
        * math.cos(phi2)
        * math.cos(dlambda)
    )

    return (
        math.degrees(
            math.atan2(y, x)
        )
        + 360.0
    ) % 360.0


def angular_difference(a, b):
    return (
        b - a + 180.0
    ) % 360.0 - 180.0


def detect_switchbacks(
    route,
    window_m,
    step_m,
    min_turn_deg=35.0,
    min_turns=3,
):
    d = route["distance_m"].to_numpy(dtype=float)
    lat = route["lat"].to_numpy(dtype=float)
    lon = route["lon"].to_numpy(dtype=float)

    if len(route) < 4:
        return pd.DataFrame()

    bearings = np.array([
        bearing_deg(
            lat[i],
            lon[i],
            lat[i + 1],
            lon[i + 1],
        )
        for i in range(len(route) - 1)
    ])

    turns = np.array([
        angular_difference(
            bearings[i],
            bearings[i + 1],
        )
        for i in range(len(bearings) - 1)
    ])

    turn_distance = d[1:-1]

    rows = []
    start = 0.0
    total = float(d[-1])

    while start + window_m <= total + 1e-9:
        end = start + window_m

        idx = (
            (turn_distance >= start)
            & (turn_distance <= end)
        )

        local_turns = turns[idx]

        significant = local_turns[
            np.abs(local_turns)
            >= min_turn_deg
        ]

        signs = np.sign(significant)

        alternations = (
            int(
                np.sum(
                    signs[1:] * signs[:-1] < 0
                )
            )
            if len(signs) >= 2
            else 0
        )

        if (
            len(significant) >= min_turns
            and alternations >= 2
        ):
            confidence = min(
                1.0,
                0.2 * len(significant)
                + 0.15 * alternations,
            )

            rows.append({
                "feature": "lacets",
                "family": "geometry",
                "start_m": float(start),
                "end_m": float(end),
                "resolution_m": float(window_m),
                "confidence": float(confidence),
                "direction_changes": int(
                    len(significant)
                ),
                "alternations": int(
                    alternations
                ),
            })

        start += step_m

    return pd.DataFrame(rows)

## 6. Exécuter un outil de détection

Cette fonction constitue l'interface entre l'agent et les outils Python.

In [ ]:
def run_detector(
    feature,
    window_m,
    step_m,
):
    detector = FEATURES[feature]["detector"]

    if detector == "slope":
        return detect_slope_feature(
            route,
            feature,
            window_m,
            step_m,
        )

    if detector == "switchbacks":
        return detect_switchbacks(
            route,
            window_m,
            step_m,
        )

    raise ValueError(
        f"Détecteur inconnu : {detector}"
    )

## 7. Résumés déterministes des détections

La couverture est la longueur de l'union des intervalles détectés divisée par la longueur de la route. Le nombre de détections brutes peut varier avec le pas, sans que cela signifie une meilleure détection.

In [ ]:
def summarize_detection(
    feature,
    result,
    window_m,
    step_m,
):
    if result.empty:
        return {
            "window_m": float(window_m),
            "step_m": float(step_m),
            "n_detections": 0,
            "detected_span_sum_m": 0.0,
            "approx_route_coverage_ratio": 0.0,
            "mean_confidence": 0.0,
        }

    intervals = result[
        ["start_m", "end_m"]
    ].sort_values("start_m")

    merged = []

    for _, row in intervals.iterrows():
        start = float(row["start_m"])
        end = float(row["end_m"])

        if not merged or start > merged[-1][1]:
            merged.append([start, end])
        else:
            merged[-1][1] = max(
                merged[-1][1],
                end,
            )

    detected_length = sum(
        end - start
        for start, end in merged
    )

    summary = {
        "window_m": float(window_m),
        "step_m": float(step_m),
        "n_detections": int(len(result)),
        "n_merged_zones": int(len(merged)),
        "detected_span_sum_m": float(
            detected_length
        ),
        "approx_route_coverage_ratio": float(
            detected_length
            / max(ROUTE_LENGTH_M, 1e-9)
        ),
        "mean_confidence": float(
            result["confidence"].mean()
        ),
    }

    if "slope_pct" in result.columns:
        summary.update({
            "mean_abs_detected_slope_pct": float(
                result["slope_pct"].abs().mean()
            ),
            "max_abs_detected_slope_pct": float(
                result["slope_pct"].abs().max()
            ),
        })

    if "direction_changes" in result.columns:
        summary.update({
            "mean_direction_changes": float(
                result["direction_changes"].mean()
            ),
            "max_direction_changes": int(
                result["direction_changes"].max()
            ),
        })

    return summary

## 8. Client Ollama

Réponse JSON structurée et température modérée pour limiter la répétition déterministe, sans garantir artificiellement des échelles différentes.


In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "llama3.1:8b"
# Une légère variabilité évite la réponse strictement déterministe ; elle ne garantit
# pas, à elle seule, des échelles différentes.
OLLAMA_TEMPERATURE = 0.35


def ask_ollama_json(prompt):
    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL,
            "messages": [
                {"role": "system", "content": (
                    "Tu es un analyste géospatial. Réponds exclusivement avec un objet JSON "
                    "valide, sans balises Markdown. Justifie brièvement les décisions "
                    "par des observations propres au phénomène et à l'itinéraire."
                )},
                {"role": "user", "content": prompt},
            ],
            "stream": False,
            "format": "json",
            "options": {"temperature": OLLAMA_TEMPERATURE},
        },
        timeout=120,
    )
    response.raise_for_status()
    content = response.json()["message"]["content"]
    decision = json.loads(content)
    if not isinstance(decision, dict):
        raise ValueError("Ollama doit renvoyer un objet JSON.")
    return decision


## 9. Valider un choix libre de l'agent

L'agent choisit librement sa valeur.

Python impose seulement :

```text
min <= window <= max
BASE_SPACING_M <= step <= window
```

afin de garantir une exécution physiquement et numériquement raisonnable.

In [ ]:
def validate_scale_choice(feature, window_m, step_m):
    minimum, maximum = FEATURES[feature]["window_bounds_m"]
    effective_max = min(float(maximum), float(ROUTE_LENGTH_M))
    if effective_max < float(minimum):
        raise ValueError(f"Parcours trop court pour {feature} : {ROUTE_LENGTH_M} m")
    window, step = float(window_m), float(step_m)
    if not (math.isfinite(window) and math.isfinite(step)):
        raise ValueError("window_m et step_m doivent être finis.")
    window = max(float(minimum), min(window, effective_max))
    step = max(float(BASE_SPACING_M), min(step, window))
    return window, step


def parse_scale_decision(feature, decision):
    if "window_m" not in decision or "step_m" not in decision:
        raise ValueError("La réponse de l'agent doit contenir window_m et step_m.")
    window, step = validate_scale_choice(
        feature, decision["window_m"], decision["step_m"]
    )
    return {
        "window_m": float(window),
        "step_m": float(step),
        "justification": str(decision.get("justification", "")),
    }


## 10. Proposition initiale sans ancrage numérique

Le schéma JSON est décrit par ses clés, sans valeurs d’exemple. Le choix reste libre à l’intérieur des bornes de sécurité.


In [ ]:
def propose_initial_scale(feature):
    cfg = FEATURES[feature]
    minimum, maximum = cfg["window_bounds_m"]
    effective_max = min(maximum, ROUTE_LENGTH_M)
    prompt = f"""
Mission : définir librement une échelle spatiale pour détecter « {feature} ».
Nature du phénomène : {cfg['description']}
Outil de mesure : {cfg['detector']} (calcul réalisé par Python, pas par toi).
Données observées sur ce parcours :
{json.dumps(ROUTE_SUMMARY, ensure_ascii=False, indent=2)}

Choisis indépendamment la portée spatiale de la fenêtre et la finesse du pas.
Raisonne sur la taille physique probable du phénomène, le relief, la longueur du
parcours, la résolution des mesures et le compromis bruit / lissage / localisation.
Distingue les variations locales, les tendances prolongées et les changements
de direction ; ne réutilise pas mécaniquement une même échelle entre phénomènes.
Ne cherche pas à produire des valeurs différentes pour le seul principe.
Aucune liste de fenêtres candidates n'est fournie.

Bornes de sécurité (pas des valeurs conseillées) :
window_m entre {minimum:.1f} et {effective_max:.1f} mètres ;
step_m entre {BASE_SPACING_M:.1f} mètres et window_m.

Renvoie exclusivement un objet JSON contenant les clés numériques "window_m",
"step_m" et la chaîne "justification". N'utilise pas de valeurs d'exemple.
"""
    return parse_scale_decision(feature, ask_ollama_json(prompt))


## 11. Comparaison multi-échelle et révision de l'agent

À chaque cycle, Python teste l'échelle courante et quatre perturbations **multiplicatives**, avec bornage et déduplication. Les essais déjà exécutés sont réutilisés (cache). Les indices de Jaccard comparent les unions d'intervalles, et non les lignes brutes. Deux résultats vides donnent une similarité **indéterminée** (`null`), jamais une stabilité parfaite.

Les coefficients relatifs sont des perturbations expérimentales, **pas** une grille fixe de fenêtres en mètres. Ils ne contraignent pas la proposition initiale ni une éventuelle révision libre du LLM.

In [ ]:
# Paramètres d'exploration relatifs : aucun candidat absolu en mètres.
RELATIVE_WINDOW_SHORT = 0.70
RELATIVE_WINDOW_LONG = 1.40
RELATIVE_STEP_SHORT = 0.65
RELATIVE_STEP_LONG = 1.50
MAX_AGENT_ITERATIONS = 3


def scale_key(window_m, step_m):
    # Déduplication numérique après application des bornes.
    return (round(float(window_m), 6), round(float(step_m), 6))


def generate_relative_trials(feature, window_m, step_m):
    variants = [
        ("current", window_m, step_m),
        ("window_shorter", window_m * RELATIVE_WINDOW_SHORT, step_m),
        ("window_longer", window_m * RELATIVE_WINDOW_LONG, step_m),
        ("step_smaller", window_m, step_m * RELATIVE_STEP_SHORT),
        ("step_larger", window_m, step_m * RELATIVE_STEP_LONG),
    ]
    unique = {}
    for label, w, s in variants:
        w, s = validate_scale_choice(feature, w, s)
        key = scale_key(w, s)
        if key not in unique:
            unique[key] = {"variant": label, "window_m": w, "step_m": s}
    return list(unique.values())


def merge_detected_intervals(result):
    if result.empty:
        return []
    intervals = sorted((float(a), float(b)) for a, b in
                       result[["start_m", "end_m"]].itertuples(index=False, name=None)
                       if math.isfinite(float(a)) and math.isfinite(float(b)) and b > a)
    merged = []
    for a, b in intervals:
        if merged and a <= merged[-1][1]:
            merged[-1][1] = max(b, merged[-1][1])
        else:
            merged.append([a, b])
    return merged


def interval_length(intervals):
    return sum(b - a for a, b in intervals)


def interval_jaccard(left, right):
    # Aucune détection des deux côtés n'apporte aucune preuve de stabilité.
    if not left and not right:
        return None
    i = j = 0
    intersection = 0.0
    while i < len(left) and j < len(right):
        a, b = left[i]
        c, d = right[j]
        intersection += max(0.0, min(b, d) - max(a, c))
        if b <= d:
            i += 1
        else:
            j += 1
    union = interval_length(left) + interval_length(right) - intersection
    return float(intersection / union) if union > 0 else None


def approximate_window_cost(window_m, step_m):
    # Nombre de fenêtres évaluées par le détecteur, y compris sans détection.
    if window_m > ROUTE_LENGTH_M:
        return 0
    return max(0, int(math.floor((ROUTE_LENGTH_M - window_m) / step_m + 1e-9)) + 1)


def evaluate_trial(feature, trial, cache):
    key = scale_key(trial["window_m"], trial["step_m"])
    if key not in cache:
        result = run_detector(feature, trial["window_m"], trial["step_m"])
        summary = summarize_detection(feature, result, trial["window_m"], trial["step_m"])
        merged = merge_detected_intervals(result)
        summary.update({
            "n_merged_zones": len(merged),
            "detected_span_sum_m": float(interval_length(merged)),
            "approx_route_coverage_ratio": float(interval_length(merged) / max(ROUTE_LENGTH_M, 1e-9)),
            "approx_window_cost": approximate_window_cost(trial["window_m"], trial["step_m"]),
        })
        cache[key] = {"result": result, "summary": summary, "intervals": merged}
    return cache[key]


def compare_trials(feature, trials, cache, cycle):
    # Chaque essai est comparé à la proposition courante et aux autres essais.
    baseline = evaluate_trial(feature, trials[0], cache)["intervals"]
    rows = []
    for idx, trial in enumerate(trials):
        entry = evaluate_trial(feature, trial, cache)
        other_jaccards = [interval_jaccard(entry["intervals"],
                            evaluate_trial(feature, other, cache)["intervals"])
                          for j, other in enumerate(trials) if j != idx]
        valid = [x for x in other_jaccards if x is not None]
        row = {
            "feature": feature, "cycle": cycle, "trial_id": f"cycle_{cycle}_trial_{idx}",
            "variant": trial["variant"], **entry["summary"],
            "jaccard_vs_current": interval_jaccard(entry["intervals"], baseline),
            "mean_pairwise_jaccard": float(np.mean(valid)) if valid else None,
            "n_valid_jaccard_comparisons": len(valid),
            "both_empty_vs_current": not entry["intervals"] and not baseline,
        }
        rows.append(row)
    return rows


def parse_review_response(feature, response, trials):
    if not isinstance(response, dict):
        raise ValueError("Révision : objet JSON attendu.")
    action = str(response.get("action", "")).strip().lower()
    justification = response.get("justification")
    if action not in {"accept", "select", "revise"}:
        raise ValueError("Révision : action attendue accept, select ou revise.")
    if not isinstance(justification, str) or not justification.strip():
        raise ValueError("Révision : justification non vide obligatoire.")
    decision = {"action": action, "justification": justification.strip()}
    if action == "select":
        trial_id = response.get("trial_id")
        ids = {trial["trial_id"] for trial in trials}
        if not isinstance(trial_id, str) or trial_id not in ids:
            raise ValueError("Révision : trial_id inconnu.")
        decision["trial_id"] = trial_id
    if action == "revise":
        if not all(k in response for k in ("window_m", "step_m")):
            raise ValueError("Révision : échelle manquante.")
        w, s = validate_scale_choice(feature, response["window_m"], response["step_m"])
        decision.update(window_m=w, step_m=s)
    return decision


In [ ]:
def review_scale(feature, current_decision, comparisons, history):
    cfg = FEATURES[feature]
    minimum, maximum = cfg["window_bounds_m"]
    effective_max = min(maximum, ROUTE_LENGTH_M)
    # JSON compact et sans valeurs fictives dans le schéma : seuls les résultats
    # réellement calculés et les bornes du parcours contiennent des nombres.
    prompt = f"""
Tu es un analyste géospatial qui révise un choix d'échelle.
Phénomène : {feature} — {cfg['description']}
Parcours : {json.dumps(ROUTE_SUMMARY, ensure_ascii=False)}
Échelle actuellement proposée : {json.dumps(current_decision, ensure_ascii=False)}
Essais réellement mesurés (identifiants trial_id) :
{json.dumps(comparisons, ensure_ascii=False, allow_nan=False)}
Historique des décisions et des cycles précédents :
{json.dumps(history, ensure_ascii=False, allow_nan=False)}

Analyse les compromis entre lissage, fragmentation, localisation et coût.
Compare les intervalles fusionnés à l'aide des indices de Jaccard disponibles.
Un Jaccard nul signale des zones disjointes ; null signale une comparaison
indéterminée. Deux essais sans détection ne prouvent PAS une stabilité.
Les détections brutes et la couverture ne sont PAS des scores de qualité.
La confiance calculée par les détecteurs n'est PAS une précision validée.
Ne sélectionne pas mécaniquement la couverture ou le nombre de détections maximal.
Une absence de détection peut être légitime. Justifie ton raisonnement avec
les différences observées et les propriétés physiques du phénomène.

Décide librement :
- action "accept" : conserver l'échelle courante ;
- action "select" : choisir un essai mesuré, avec sa clé "trial_id" exacte ;
- action "revise" : inventer une nouvelle fenêtre et un nouveau pas,
  avec "window_m" et "step_m" numériques, sans grille imposée.
Toute réponse doit contenir "action" et "justification" non vide.
Bornes de sécurité : window_m entre {minimum:.1f} et {effective_max:.1f} mètres,
step_m entre {BASE_SPACING_M:.1f} mètres et window_m.
Renvoie uniquement un objet JSON, sans exemple ni commentaire extérieur.
"""
    return parse_review_response(feature, ask_ollama_json(prompt), comparisons)


## 12. Boucle de décision multi-échelle (trois cycles au maximum)

Chaque cycle teste la proposition courante et ses variantes relatives. Une décision `select` récupère exactement le résultat déjà calculé. Une révision au dernier cycle est enregistrée mais **non appliquée**, car elle n'a pas été évaluée. Si Ollama échoue ou renvoie un JSON invalide, l'échelle courante déjà testée est conservée avec un statut explicite `fallback`, sans attribuer cette décision au LLM.

In [ ]:
MAX_AGENT_ITERATIONS = 3

def run_scale_agent(feature, max_iterations=MAX_AGENT_ITERATIONS):
    if not isinstance(max_iterations, int) or max_iterations < 1 or max_iterations > MAX_AGENT_ITERATIONS:
        raise ValueError("max_iterations doit être compris dans la limite des cycles autorisés.")
    initial = propose_initial_scale(feature)
    current = dict(initial)
    cache = {}
    history = []
    all_comparisons = []
    final_key = None
    accepted = False
    final_reason = ""

    for cycle in range(1, max_iterations + 1):
        trials = generate_relative_trials(feature, current["window_m"], current["step_m"])
        comparisons = compare_trials(feature, trials, cache, cycle)
        all_comparisons.extend(comparisons)
        cycle_record = {
            "iteration": cycle, "proposal": dict(current),
            "trials": comparisons, "review": None,
        }
        history.append(cycle_record)
        print(f"\n{feature} — cycle {cycle} : {len(comparisons)} essais distincts")
        display(pd.DataFrame(comparisons)[[
            "trial_id", "variant", "window_m", "step_m", "n_detections",
            "n_merged_zones", "approx_route_coverage_ratio", "mean_confidence",
            "approx_window_cost", "jaccard_vs_current", "mean_pairwise_jaccard",
        ]])

        if cycle == max_iterations:
            # Au dernier cycle, le LLM peut encore sélectionner un essai évalué,
            # mais une révision non évaluée ne peut pas être retenue.
            print("Dernier cycle : seule une échelle déjà mesurée peut être retenue.")
        try:
            review = review_scale(feature, current, comparisons, history[:-1])
        except (ValueError, KeyError, TypeError, requests.RequestException, json.JSONDecodeError) as exc:
            # Échec explicite, reproductible : conserver l'échelle courante
            # effectivement évaluée, sans prétendre à une acceptation par le LLM.
            review = {"action": "fallback", "justification": f"Réponse LLM invalide ou indisponible : {type(exc).__name__}: {exc}"}

        cycle_record["review"] = review
        action = review["action"]
        final_reason = review["justification"]
        if action == "select":
            selected = next(row for row in comparisons if row["trial_id"] == review["trial_id"])
            final_key = scale_key(selected["window_m"], selected["step_m"])
            accepted = True
            break
        if action == "accept":
            final_key = scale_key(current["window_m"], current["step_m"])
            accepted = True
            break
        if action == "fallback":
            final_key = scale_key(current["window_m"], current["step_m"])
            break
        if action == "revise":
            revised_key = scale_key(review["window_m"], review["step_m"])
            current_key = scale_key(current["window_m"], current["step_m"])
            if revised_key == current_key:
                cycle_record["review_status"] = "unchanged_revision_kept_current"
                final_key = current_key
                break
            if cycle == max_iterations:
                # La proposition non testée est enregistrée, mais jamais exportée.
                cycle_record["review_status"] = "untested_revision_rejected_at_cycle_limit"
                final_key = current_key
                break
            current = {"window_m": review["window_m"],
                       "step_m": review["step_m"], "justification": final_reason}
            continue
        raise RuntimeError(f"Action inattendue : {action}")

    if final_key is None or final_key not in cache:
        raise RuntimeError("Aucune échelle finale effectivement testée.")
    final = {"window_m": final_key[0], "step_m": final_key[1],
             "justification": final_reason}
    return {
        "feature": feature, "initial_decision": initial,
        "final_decision": final, "history": history,
        "comparisons": all_comparisons,
        "result": cache[final_key]["result"],
        "accepted_by_agent": accepted,
        "n_iterations": len(history),
        "final_status": history[-1]["review"]["action"],
    }


## 13. Lancer l'Agent IA sur les caractéristiques

Pour commencer, on conserve les cinq caractéristiques actuelles.

À terme, cette liste pourra elle aussi devenir un objet de décision agentique.

In [ ]:
FEATURES_TO_ANALYZE = [
    "forte_pente",
    "pente_douce",
    "montee",
    "descente",
    "lacets",
]

agent_runs = {}

for feature in FEATURES_TO_ANALYZE:
    print("\n" + "=" * 80)
    print("AGENT —", feature)

    agent_runs[feature] = (
        run_scale_agent(feature)
    )

## 14. Inspecter les fenêtres choisies par l'Agent

Cette table est importante pour les expériences futures.

Elle permettra notamment de comparer :

```text
fenêtre choisie par l'agent
VS
fenêtre considérée pertinente par un expert
VS
qualité finale de détection
```

In [ ]:
scale_rows = []
for feature, run in agent_runs.items():
    final = run["final_decision"]
    scale_rows.append({
        "feature": feature, "detector": FEATURES[feature]["detector"],
        "initial_window_m": run["initial_decision"]["window_m"],
        "initial_step_m": run["initial_decision"]["step_m"],
        "window_m": final["window_m"], "step_m": final["step_m"],
        "n_iterations": run["n_iterations"],
        "n_comparisons": len(run["comparisons"]),
        "accepted_by_agent": run["accepted_by_agent"],
        "final_status": run["final_status"],
        "justification": final.get("justification", ""),
    })
agent_scale_df = pd.DataFrame(scale_rows)
display(agent_scale_df)

# Tableau global : une ligne par essai, même si aucune détection.
agent_comparisons_df = pd.DataFrame([
    row for run in agent_runs.values() for row in run["comparisons"]
])
display(agent_comparisons_df)


## 15. Standardiser les annotations atomiques

La sortie reste compatible avec l'algorithme du notebook 04.

Chaque observation conserve notamment :

```text
resolution_m = fenêtre réellement choisie par l'Agent IA
```

In [ ]:
STANDARD_COLUMNS = {
    "feature",
    "family",
    "start_m",
    "end_m",
    "resolution_m",
    "confidence",
}

atomic_annotations = []

for feature, run in agent_runs.items():
    result = run["result"]

    if result.empty:
        continue

    final_decision = (
        run["final_decision"]
    )

    for _, row in result.iterrows():
        metrics = {
            "agent_window_m": float(
                final_decision[
                    "window_m"
                ]
            ),
            "agent_step_m": float(
                final_decision[
                    "step_m"
                ]
            ),
            "agent_iterations": int(
                len(run["history"])
            ),
            "agent_accepted_scale": bool(
                run[
                    "accepted_by_agent"
                ]
            ),
        }

        for column in result.columns:
            if column in STANDARD_COLUMNS:
                continue

            value = row[column]

            if pd.isna(value):
                continue

            if hasattr(value, "item"):
                value = value.item()

            metrics[column] = value

        atomic_annotations.append({
            "feature": str(
                row["feature"]
            ),
            "family": str(
                row["family"]
            ),
            "start_m": float(
                row["start_m"]
            ),
            "end_m": float(
                row["end_m"]
            ),
            "resolution_m": float(
                row["resolution_m"]
            ),
            "confidence": float(
                row["confidence"]
            ),
            "metrics": metrics,
        })

print(
    "Annotations atomiques :",
    len(atomic_annotations)
)

## 16. Exports vers le notebook 04 et journal multi-échelle

Les annotations atomiques conservent leur schéma (`feature`, `family`, `start_m`, `end_m`, `resolution_m`, `confidence`, `metrics`). Les décisions JSON incluent la proposition initiale, tous les essais et les révisions. Le CSV supplémentaire contient une ligne par essai de chaque cycle. **Aucun export ne certifie la qualité des détections.**

In [ ]:
OUTPUT_DIR = Path(
    "data/output"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

json_path = (
    OUTPUT_DIR
    / "agent1_annotations_atomic.json"
)

csv_path = (
    OUTPUT_DIR
    / "agent1_annotations_atomic.csv"
)

decision_path = (
    OUTPUT_DIR
    / "agent1_scale_decisions.json"
)

with json_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        atomic_annotations,
        f,
        ensure_ascii=False,
        indent=2,
    )

pd.DataFrame([
    {
        **{
            k: v
            for k, v in annotation.items()
            if k != "metrics"
        },
        "metrics_json": json.dumps(
            annotation["metrics"],
            ensure_ascii=False,
        ),
    }
    for annotation in atomic_annotations
]).to_csv(
    csv_path,
    index=False,
)

serializable_runs = {}

for feature, run in agent_runs.items():
    serializable_runs[feature] = {
        "initial_decision": run["initial_decision"],
        "comparisons": run["comparisons"],
        "n_iterations": run["n_iterations"],
        "final_status": run["final_status"],
        "final_decision": (
            run["final_decision"]
        ),
        "history": run["history"],
        "accepted_by_agent": bool(
            run[
                "accepted_by_agent"
            ]
        ),
    }

with decision_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        serializable_runs,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Exports :")
print("-", json_path)
print("-", csv_path)
print("-", decision_path)
comparison_path = OUTPUT_DIR / "agent1_scale_comparisons.csv"
agent_comparisons_df.to_csv(comparison_path, index=False)
print("-", comparison_path)


# Portée et limites scientifiques de la V4

- **Autonomie :** le LLM propose une échelle librement, peut accepter, sélectionner un essai mesuré ou proposer une nouvelle échelle. Les variantes sont relatives à sa proposition, jamais une liste fixe de fenêtres absolues.
- **Stabilité :** un Jaccard élevé indique que deux essais recouvrent des intervalles semblables. Ce n'est pas une validation terrain. Deux essais vides ont un Jaccard indéterminé, et non égal à un.
- **Couverture :** la proportion de distance annotée peut augmenter par lissage ou par multiplication de faux positifs. Elle ne doit pas être maximisée automatiquement.
- **Confiance :** les valeurs existantes sont des heuristiques internes aux détecteurs, pas des probabilités calibrées.
- **Qualité réelle :** impossible de conclure à l'optimalité sans vérité terrain et mesures de précision/rappel, analyse des erreurs et robustesse sur plusieurs parcours.
- **Coût :** le nombre de fenêtres est une approximation du travail géométrique, pas un temps CPU mesuré. Le détecteur de lacets peut aussi dépendre du nombre de points.
- **Traçabilité :** les décisions JSON, les variantes et les comparaisons CSV permettent d'auditer la sélection ; un repli après erreur LLM est explicitement marqué.
- **Reproductibilité :** les décisions LLM peuvent varier avec la température et le modèle Ollama. Conserver le modèle, sa version, le GPX et les paramètres d'exécution pour une expérience contrôlée.
